# Notion Knowledge Base with RAG

Most team knowledge lives in Notion — specs, runbooks, meeting notes, onboarding docs. This notebook turns your Notion workspace into a searchable knowledge base using SynapseKit's `NotionLoader` and RAG pipeline. Ask questions in natural language and get answers grounded in your actual documentation.

**What you'll build:** A RAG system that loads pages from a Notion database, chunks and indexes them, and answers questions with citations back to the source pages.

**Time:** ~20 min | **Difficulty:** Intermediate

**What you'll learn:**
- Authenticate and load pages from Notion using `NotionLoader`
- Convert Notion blocks to plain text for indexing
- Chunk documents and build a vector index with `RAG`
- Query the index and return answers with source citations
- Incrementally refresh the index as Notion pages are updated

## Prerequisites & Setup

You need a Notion integration token from https://www.notion.so/my-integrations. After creating the integration, share each Notion page or database with it: open the page → Share → Invite your integration.

In [ ]:
!pip install "synapsekit[openai,notion]" -q

In [ ]:
import os

# os.environ["OPENAI_API_KEY"] = "sk-..."
# os.environ["NOTION_TOKEN"] = "secret_..."   # Notion integration token

## Step 1: Load pages from Notion

`NotionLoader` fetches pages from a Notion database or a list of page IDs. It converts all block types (paragraphs, headings, bullets, code, tables) to plain text.

In [ ]:
import asyncio
from synapsekit.loaders import NotionLoader

# NotionLoader fetches pages from a Notion database or a list of page IDs.
loader = NotionLoader(
    token="NOTION_TOKEN",        # Resolved from env var automatically
    # Option A: load an entire database
    database_id="your-database-id-here",
    # Option B: load specific pages
    # page_ids=["page-id-1", "page-id-2"],
)

async def load_docs():
    docs = await loader.aload()
    print(f"Loaded {len(docs)} documents from Notion")
    for doc in docs[:3]:
        print(f"  - {doc.metadata['title']} ({len(doc.content)} chars)")
    return docs

docs = await load_docs()

## Step 2: Index documents with RAG

`RAG` handles chunking, embedding, and indexing automatically. `chunk_size=512` tokens is a good starting point for prose documentation.

In [ ]:
from synapsekit import RAG
from synapsekit.llms.openai import OpenAILLM

rag = RAG(
    llm=OpenAILLM(model="gpt-4o-mini"),
    chunk_size=512,
    chunk_overlap=64,
    # By default uses an in-memory vector store -- swap for a persistent one in production
)

async def build_index(docs):
    print("Building vector index...")
    await rag.aadd_documents(docs)
    print(f"Indexed {rag.document_count} chunks from {len(docs)} documents.")

await build_index(docs)

## Step 3: Answer questions with citations

`result.sources` contains the most relevant chunks with metadata, enabling grounded answers with page links.

In [ ]:
async def ask(question: str) -> dict:
    """Query the knowledge base and return an answer with source citations."""
    result = await rag.aquery(question)

    sources = [
        {
            "title":   chunk.metadata.get("title", "Untitled"),
            "url":     chunk.metadata.get("url", ""),
            "excerpt": chunk.content[:200],
        }
        for chunk in result.sources
    ]

    return {
        "question": question,
        "answer":   result.answer,
        "sources":  sources,
    }

def format_answer(result: dict):
    print(f"\nQ: {result['question']}")
    print(f"\nA: {result['answer']}")
    print(f"\nSources ({len(result['sources'])}):")
    for src in result["sources"]:
        print(f"  - {src['title']}")
        if src["url"]:
            print(f"    {src['url']}")

# Try it
result = await ask("What is our on-call rotation policy?")
format_answer(result)

## Step 4: Incremental index refresh

Reload only pages edited since a given time and update the index without a full re-index.

In [ ]:
import asyncio
from datetime import datetime, timedelta

async def refresh_index(since: datetime | None = None):
    """Reload pages edited since `since` and update the index.

    Call this on a schedule (e.g. every 15 minutes) to keep the index fresh
    without a full re-index every time.
    """
    since = since or (datetime.utcnow() - timedelta(hours=1))

    # NotionLoader filters by last_edited_time when since is provided
    updated_docs = await loader.aload(edited_after=since)

    if not updated_docs:
        print(f"No Notion pages updated since {since.isoformat()}")
        return

    print(f"Refreshing {len(updated_docs)} updated pages...")
    # remove_by_source removes old chunks for the same page before re-adding
    for doc in updated_docs:
        await rag.aremove_source(doc.metadata["id"])
    await rag.aadd_documents(updated_docs)
    print("Index refreshed.")

print("refresh_index() defined. Call periodically to keep the index fresh.")

## Complete Working Example

A full pipeline from loading Notion pages to answering questions with citations.

In [ ]:
import asyncio
from synapsekit import RAG
from synapsekit.llms.openai import OpenAILLM
from synapsekit.loaders import NotionLoader

async def main():
    loader = NotionLoader(
        token="NOTION_TOKEN",
        database_id="your-database-id-here",
    )

    rag = RAG(
        llm=OpenAILLM(model="gpt-4o-mini"),
        chunk_size=512,
        chunk_overlap=64,
    )

    # Load and index
    print("Loading Notion workspace...")
    docs = await loader.aload()
    print(f"Loaded {len(docs)} pages.")

    print("Building index...")
    await rag.aadd_documents(docs)
    print(f"Indexed {rag.document_count} chunks.\n")

    # Interactive Q&A loop
    questions = [
        "What is our on-call rotation policy?",
        "How do I set up the development environment?",
        "What were the key decisions from last month's architecture review?",
    ]

    for question in questions:
        result = await rag.aquery(question)
        print(f"Q: {question}")
        print(f"A: {result.answer[:200]}...")
        print(f"   Sources: {', '.join(c.metadata.get('title','?') for c in result.sources[:2])}")
        print()

asyncio.run(main())

## Variations

In [ ]:
# Use a persistent vector store (Chroma):
# from synapsekit.vectorstores import ChromaVectorStore
# rag = RAG(
#     llm=OpenAILLM(model="gpt-4o-mini"),
#     vector_store=ChromaVectorStore(path="./notion_index"),
# )

# Filter by Notion database properties:
# docs = await loader.aload(
#     filter={"property": "Status", "select": {"equals": "Published"}}
# )

# Multi-workspace support:
# loaders = [
#     NotionLoader(token="TOKEN_1", database_id="workspace-1-db"),
#     NotionLoader(token="TOKEN_2", database_id="workspace-2-db"),
# ]
# all_docs = []
# for loader in loaders:
#     all_docs.extend(await loader.aload())
# await rag.aadd_documents(all_docs)

print("Variation examples shown above as comments.")

## Next Steps

- **[Slack Q&A Bot](slack-qa-bot.ipynb)** — expose your Notion knowledge base as a Slack bot
- **[YouTube Video Summarizer](youtube-summarizer.ipynb)** — index video transcripts alongside Notion pages
- **[LLM Provider Comparison](../llms/provider-comparison.ipynb)** — swap in a cheaper LLM for the Q&A step

**Troubleshooting:**
- `APIResponseError: Could not find database`: Share the database with your integration (open database → `...` → Connections → add your integration)
- Answers reference outdated information: Set up a periodic refresh with `loader.aload(edited_after=last_refresh_time)`
- Chunks are too large and miss detail: Reduce `chunk_size` to 256 and increase `top_k` in `rag.aquery(top_k=8)`